# Data Exploration Notebook

This notebook provides an interactive environment for exploring the Google Cluster Workload Traces dataset and understanding data skew patterns.

In [ ]:
# Import necessary libraries
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
sys.path.insert(0, str(Path().resolve().parent / "src"))

# Import project modules
from data_loader import load_task_events, get_sample_data
from preprocessing import clean_task_events, extract_task_runtimes
from feature_engineering import extract_pre_execution_features, encode_categorical_features
from skew_labeling import label_jobs_from_task_runtimes, get_skew_statistics

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Load Dataset

Load the task_events.csv file. For faster exploration, you can use `get_sample_data()` with a limited number of rows.

In [ ]:
# Load dataset (use get_sample_data(n_rows=100000) for faster exploration)
try:
    # For full dataset:
    df = load_task_events()
    
    # For sample (uncomment to use):
    # df = get_sample_data(n_rows=100000)
    
    print(f"Dataset loaded: {len(df):,} rows, {len(df.columns)} columns")
    print(f"\nColumns: {list(df.columns)}")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Please ensure task_events.csv is in data/raw/ directory")

In [ ]:
# Display basic information about the dataset
print("Dataset Info:")
print(df.info())
print("\nFirst few rows:")
df.head()

In [ ]:
# Basic statistics
print("Basic Statistics:")
df.describe()

## 2. Data Cleaning and Preprocessing

In [ ]:
# Clean the data
df_clean = clean_task_events(df)
print(f"Cleaned dataset: {len(df_clean):,} rows")

In [ ]:
# Extract task runtimes
df_runtimes = extract_task_runtimes(df_clean)
print(f"Tasks with runtimes: {len(df_runtimes):,}")
print(f"\nRuntime statistics:")
print(df_runtimes['runtime'].describe())

## 3. Feature Engineering - Pre-Execution Features

In [ ]:
# Extract pre-execution features from submit events
pre_exec_features = extract_pre_execution_features(df_clean)
pre_exec_features = encode_categorical_features(pre_exec_features)

print(f"Job-level pre-exec features: {len(pre_exec_features):,} jobs")
print(f"\nFeature columns: {list(pre_exec_features.columns)}")
pre_exec_features.head()

In [ ]:
# Job-level feature statistics
print("Job-Level Feature Statistics:")
pre_exec_features.describe()

## 4. Skew Labeling

In [ ]:
# Label skewed jobs from runtime stats
labels = label_jobs_from_task_runtimes(df_runtimes)
job_labeled = pre_exec_features.merge(labels, on="job_id", how="inner")

# Get skew statistics
stats = get_skew_statistics(job_labeled)
print("\nSkew Statistics:")
for key, value in stats.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value:,}")

## 5. Visualizations

In [ ]:
# Distribution of skew labels
plt.figure(figsize=(8, 6))
job_labeled['is_skewed'].value_counts().plot(kind='bar', color=['skyblue', 'salmon'])
plt.title('Distribution of Skewed vs Non-Skewed Jobs')
plt.xlabel('Is Skewed (0=No, 1=Yes)')
plt.ylabel('Number of Jobs')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Compare max_task_runtime for skewed vs non-skewed jobs
plt.figure(figsize=(10, 6))
skewed = job_labeled[job_labeled['is_skewed'] == 1]['max_task_runtime']
non_skewed = job_labeled[job_labeled['is_skewed'] == 0]['max_task_runtime']

plt.hist(non_skewed, bins=50, alpha=0.7, label='Non-Skewed', color='skyblue', density=True)
plt.hist(skewed, bins=50, alpha=0.7, label='Skewed', color='salmon', density=True)
plt.xlabel('Max Task Runtime')
plt.ylabel('Density')
plt.title('Distribution of Max Task Runtime: Skewed vs Non-Skewed Jobs')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: avg_task_runtime vs max_task_runtime
plt.figure(figsize=(10, 6))
skewed_jobs = job_labeled[job_labeled['is_skewed'] == 1]
non_skewed_jobs = job_labeled[job_labeled['is_skewed'] == 0]

plt.scatter(non_skewed_jobs['avg_task_runtime'], non_skewed_jobs['max_task_runtime'], 
           alpha=0.5, label='Non-Skewed', s=20, color='skyblue')
plt.scatter(skewed_jobs['avg_task_runtime'], skewed_jobs['max_task_runtime'], 
           alpha=0.5, label='Skewed', s=20, color='salmon')

# Add line: max = 2 * avg (skew threshold)
x_line = np.linspace(0, job_labeled['avg_task_runtime'].max(), 100)
y_line = 2 * x_line
plt.plot(x_line, y_line, 'r--', label='Skew Threshold (max = 2 * avg)', linewidth=2)

plt.xlabel('Average Task Runtime')
plt.ylabel('Max Task Runtime')
plt.title('Average vs Max Task Runtime: Skewed vs Non-Skewed Jobs')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(10, 8))
feature_cols = ['num_tasks', 'avg_task_runtime', 'max_task_runtime', 
                'std_task_runtime', 'scheduling_class', 'priority', 'is_skewed']
corr_matrix = job_labeled[feature_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: Feature distributions by skew label
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
features = ['num_tasks', 'avg_task_runtime', 'max_task_runtime', 
            'std_task_runtime', 'scheduling_class', 'priority']

for idx, feature in enumerate(features):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    job_labeled.boxplot(column=feature, by='is_skewed', ax=ax)
    ax.set_title(f'{feature} by Skew Label')
    ax.set_xlabel('Is Skewed')
    ax.set_ylabel(feature)
    ax.get_figure().suptitle('')  # Remove default title

plt.tight_layout()
plt.show()

## 6. Summary Statistics by Skew Label

In [ ]:
# Compare statistics for skewed vs non-skewed jobs
print("Statistics for Non-Skewed Jobs:")
print(job_labeled[job_labeled['is_skewed'] == 0][['num_tasks', 'avg_task_runtime', 
                                                    'max_task_runtime', 'std_task_runtime']].describe())
print("\nStatistics for Skewed Jobs:")
print(job_labeled[job_labeled['is_skewed'] == 1][['num_tasks', 'avg_task_runtime', 
                                                   'max_task_runtime', 'std_task_runtime']].describe())